In [2]:
import os
import openai
from openai import OpenAI

In [ ]:
client = OpenAI()

In [6]:
response = client.chat.completions.create(
    model='gpt-4.1',
    messages=[{"role": "user", "content": "how to find helper information of a function in Jupyter Notebook?"}]
)

In [7]:
response.choices[0].message.content

'To find helper information (such as the **docstring** or **documentation**) of a function in **Jupyter Notebook**, you can use several convenient methods:\n\n### 1. Question Mark (`?`)\nType the function name followed by a **question mark** and run the cell:\n```python\nprint?\n```\nor\n```python\nlen?\n```\nThis will show a popup (or a panel at the bottom) with information about the function, including its docstring, signature, etc.\n\n### 2. Double Question Mark (`??`)\nDouble question marks often display **even more detail,** sometimes including the source code if available:\n```python\nprint??\n```\n\n### 3. Tab Completion\nType the function name and then **open parenthesis** and press **`Shift + Tab`**:\n```python\nprint(   # <-- place cursor inside parentheses and press Shift + Tab\n```\nThis will show a tooltip with function signature and basic documentation. **Press `Shift + Tab` quickly twice or more** for more detailed info.\n\n### 4. `help()` Function\nCall the built-in `he

In [4]:
import json
import minsearch

In [5]:
with open('documents.json','rt') as f:
    docs_raw = json.load(f)

In [6]:
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [7]:
print(len(documents))
documents[0]

948


{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [8]:
index = minsearch.Index(
    text_fields=['question','text','section'],
    keyword_fields=['course']
)

index.fit(documents)

In [9]:
q = 'the course has already started, can I still enroll?'

In [10]:
boost = {'question':3.0, 'section':0.5}

results = index.search(
    query=q,
    filter_dict={'course':'data-engineering-zoomcamp'},
    boost_dict=boost,
    num_results=5
)

In [11]:
results

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 202

In [12]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [13]:
prompt = build_prompt(query=q, search_results=results)

In [14]:
def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [15]:
answer = llm(prompt=prompt)

In [16]:
answer

'Yes, you can still enroll in the course even after it has started. You are eligible to submit homework, but make sure to adhere to the deadlines for the final projects.'